[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C3/notebooks/leky.ipynb)

# Dva léky: táž úloha v CVXPY, v `minimize` a kontrola KKT

Úloha, která se právě spočítala na tabuli. Dva léky A a B mají ideální dávky
4 mg a 2 mg, odchylka se trestá kvadraticky a součet dávek nesmí kvůli
ledvinám překročit 4 mg:

$$
\begin{aligned}
\text{minimize}_{\mathbf x}\quad & f(\mathbf x) = (x_1-4)^2 + (x_2-2)^2\\
\text{subject to}\quad & g(\mathbf x) = x_1 + x_2 - 4 \le 0
\end{aligned}
$$

Na tabuli vyšlo $\mathbf x^\star = (3;\ 1)$ mg, $f^\star = 2$ a multiplikátor
$\mu = 2$. Tahle čísla jsou měřítko: kód, který je nepotvrdí, je špatně.

Doplňte **tři místa `# TODO`**: model v CVXPY, omezení pro `minimize`
a rezidua KKT. Zbytek je připravený.

| na tabuli | v kódu |
|---|---|
| $\mathbf x$ | `cp.Variable` |
| $f(\mathbf x)$ | `cp.sum_squares` |
| $\mu$ | `ledviny.dual_value` |
| $g(\mathbf x)\le 0$ v `minimize` | `{"type": "ineq", "fun": ...}` |

---

*Čísla, se kterými se notebook porovnává, počítá skript [`kod/cisla_leky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C3/kod/cisla_leky.py).*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import numpy as np
from scipy.optimize import minimize

CIL = np.array([4.0, 2.0])     # ideální dávky léku A a B [mg]
LIMIT = 4.0                    # společná zátěž ledvin: x1 + x2 <= 4 [mg]

## Krok 1 — CVXPY *(díra 1)*

In [ ]:
# TODO 1: přepište úlohu z tabule do CVXPY.
#   proměnná ... dvě dávky v mg
#   účelová  ... součet čtverců odchylek od CIL  (cp.sum_squares)
#   omezení  ... součet dávek nejvýš LIMIT
x = ...         # cp.Variable(...)
ledviny = ...   # nerovnice pro součet dávek
omezeni = [ledviny]
uloha = cp.Problem(cp.Minimize(...), omezeni)

In [ ]:
print("je to konvexní úloha (DCP)?", uloha.is_dcp())
uloha.solve(solver=cp.CLARABEL)
x_cvxpy = x.value
mu_cvxpy = float(ledviny.dual_value)

print(f"x* = ({x_cvxpy[0]:.4f}; {x_cvxpy[1]:.4f}) mg,  f* = {uloha.value:.4f},  mu = {mu_cvxpy:.4f}")
assert np.allclose(x_cvxpy, [3, 1], atol=1e-6) and abs(mu_cvxpy - 2) < 1e-6, "nesedí s tabulí"

`is_dcp()` = **True** znamená, že CVXPY z pravidel DCP prokázalo konvexitu, takže
KKT podmínky jsou **postačující** a nalezené minimum je globální. **False** jen říká,
že to CVXPY z pravidel nepoznalo, ne že úloha konvexní není.

## Krok 2 — `scipy.optimize.minimize` *(díra 2)*

Proč i `minimize`? CVXPY přijme jen úlohy, u kterých umí dokázat konvexitu
(pravidla DCP). `minimize` vezme libovolnou funkci, zato nic negarantuje
a multiplikátory nevrací. Obecný solver nezná CVXPY výrazy, dostane obyčejné
funkce v Pythonu.

In [ ]:
# TODO 2: zapište omezení x1 + x2 <= 4 tak, jak ho chce minimize.
# Pozor: typ "ineq" znamená fun(x) >= 0, tedy OPAČNĚ než g(x) <= 0 z přednášky.
omezeni_scipy = {"type": "ineq", "fun": lambda x: ...}

In [ ]:
res = minimize(lambda x: np.sum((x - CIL) ** 2), x0=[0.0, 0.0],
               method="SLSQP", constraints=[omezeni_scipy])
x_scipy = res.x
print(f"x* = ({x_scipy[0]:.4f}; {x_scipy[1]:.4f}) mg,  f* = {res.fun:.4f}")
assert np.allclose(x_scipy, x_cvxpy, atol=1e-5), "minimize a CVXPY se rozešly"

## Krok 3 — kontrola KKT *(díra 3)*

Čtyři podmínky z přednášky převedeme na čtyři čísla, která mají v optimu vyjít
nula: $\lVert\nabla f+\mu\nabla g\rVert$, překročení limitu $\max(0, g)$,
zápornost $\max(0,-\mu)$ a $\lvert\mu\, g\rvert$. Gradienty
$\nabla f = 2(\mathbf x - \mathbf x_{\text{cil}})$ a $\nabla g = (1;\ 1)$ jsou připravené.

In [ ]:
def g(x):
    return x[0] + x[1] - LIMIT        # g(x) <= 0

def grad_f(x):
    return 2 * (x - CIL)

grad_g = np.array([1.0, 1.0])

In [ ]:
def kkt(x, mu):
    """Vrátí čtyři čísla; v bodě splňujícím KKT jsou všechna ≈ 0."""
    # TODO 3: doplňte reziduum stacionarity a komplementarity.
    stacionarita = ...        # velikost vektoru ∇f(x) + mu·∇g   (np.linalg.norm)
    pripustnost = max(0.0, g(x))      # o kolik je limit překročen
    dualni = max(0.0, -mu)            # o kolik je mu záporné
    komplementarita = ...     # |mu · g(x)|
    return stacionarita, pripustnost, dualni, komplementarita

In [ ]:
kandidati = {
    "CVXPY": (x_cvxpy, mu_cvxpy),
    "minimize": (x_scipy, 2.0),   # minimize multiplikátor nevrací, bereme ho z tabule
    "(3; 1), mu = 1": (np.array([3.0, 1.0]), 1.0),
    "(4; 2), mu = 0": (np.array([4.0, 2.0]), 0.0),
}
print("kandidát:  stacionarita, přípustnost, mu >= 0, komplementarita")
for jmeno, (xk, mk) in kandidati.items():
    print(jmeno + ":", np.round(kkt(xk, mk), 4))

assert max(kkt(x_cvxpy, mu_cvxpy)) < 1e-6, "řešení CVXPY nesplňuje KKT"
assert kkt(np.array([3.0, 1.0]), 1.0)[0] > 1, "špatný mu musí stacionaritu porušit"
assert kkt(np.array([4.0, 2.0]), 0.0)[1] > 1, "minimum bez omezení (4; 2) porušuje limit"

První dva řádky jsou nuly až na zaokrouhlení. Třetí má dobrý bod, ale špatnou
cenu: gradienty se nevyruší. Čtvrtý je minimum bez omezení — stacionarita
platí, ale limit je překročený o 2 mg. **KKT se ověřují všechny čtyři najednou.**